In [ ]:
import pandas as pd
import numpy as np
import os

SAVE_DIR = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본"
LC_COLS  = ['buildings', 'greenspace', 'road struc', 'river zone']

# ── 로드 ─────────────────────────────────────────────────────────────────
lc         = pd.read_csv("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/토지피복데이터/grid_landcover_percentage.csv")
grid_basic = pd.read_csv(os.path.join(SAVE_DIR, '격자_250m_4326.csv'))

print(f'grid_basic: {grid_basic.shape}')
print(f'토지피복:   {lc.shape}')
print(f'컬럼: {LC_COLS}')

# ── CELL_ID 기준 병합 (100% 매칭 확인됨) ─────────────────────────────────
merged = grid_basic[['fid', 'CELL_ID', 'lon', 'lat']].merge(
    lc[['CELL_ID'] + LC_COLS], on='CELL_ID', how='left'
)

assert merged['buildings'].isna().sum() == 0, '매칭 실패한 격자 있음!'
print(f'\n병합 완료: {merged.shape}  (결측치 없음)')
print(merged[['CELL_ID', 'lat', 'lon'] + LC_COLS].head(3).to_string())

# ── [G, 4] 배열로 저장 (정적 데이터) ─────────────────────────────────────
lc_arr = merged[LC_COLS].values.astype(np.float32)   # [10125, 4]
print(f'\nlc_arr shape: {lc_arr.shape}  dtype: {lc_arr.dtype}')
print('각 변수 범위 (면적 비율 %):')
for i, col in enumerate(LC_COLS):
    print(f'  {col:12s}: {lc_arr[:, i].min():.2f} ~ {lc_arr[:, i].max():.2f}')

np.save(os.path.join(SAVE_DIR, 'landcover_static.npy'), lc_arr)
np.save(os.path.join(SAVE_DIR, 'landcover_cols.npy'),
        np.array([c.replace(' ', '_') for c in LC_COLS]))

# 통합 CSV 업데이트 (기존 격자_250m_4326_with_lur.csv에 토지피복 추가)
existing = pd.read_csv(os.path.join(SAVE_DIR, '격자_250m_4326_with_lur.csv'))
updated  = existing.merge(merged[['CELL_ID'] + LC_COLS].rename(
    columns={c: c.replace(' ', '_') for c in LC_COLS}), on='CELL_ID', how='left')
updated.to_csv(os.path.join(SAVE_DIR, '격자_250m_4326_with_lur.csv'), index=False)

print(f'\n저장 완료 → {SAVE_DIR}/')
print(f'  landcover_static.npy       {lc_arr.shape}  (buildings, greenspace, road_struc, river_zone)')
print(f'  landcover_cols.npy         컬럼명 배열')
print(f'  격자_250m_4326_with_lur.csv  토지피복 컬럼 추가됨 ({updated.shape})')